# Operations & Reliability

## What's covered

- **Why operations decides reliability** — design is necessary, operations is sufficient
- **The three pillars of observability** — metrics, logs, distributed traces
- **SLIs, SLOs, and error budgets** — measuring "is the system meeting its promises?"
- **Deployment strategies** — blue-green, canary, rolling, and feature flags
- **Failure modes and graceful degradation** — designing for "what happens when X fails?"
- **Load shedding and backpressure** — how the system stays up under overload


## Why operations is what reliability actually depends on

The first six notebooks of this series describe what to build. This notebook describes **how to run what you built**. The two are not the same skill, and the second is what mostly decides whether the system meets its uptime, latency, and correctness goals.

A famous Google SRE-team observation: **most outages are caused by changes**. Not hardware failures. Not novel network partitions. Configuration changes, deploys, schema migrations, traffic shifts. The interesting failures that take down well-designed systems happen *because someone, somewhere, changed something*. Operations is the discipline of doing that safely and noticing fast when it goes wrong.

Three categories of work cover most of operations:

- **Observability** — knowing what the system is doing right now and historically. Without this, every other operational practice is guessing.
- **Deployment safety** — changing the system without taking it down. Includes deployment strategies, feature flags, automated rollback.
- **Failure handling** — what the system does when something breaks. Graceful degradation, load shedding, runbooks, post-incident review.

This notebook walks each in turn. The patterns from the earlier notebooks reappear — circuit breakers (notebook 06), eventual consistency (notebook 04), tail latency (notebook 01) — because operations is where those patterns prove themselves.


## The three pillars of observability

You cannot operate what you cannot see. Modern observability rests on three pillars:

- **Metrics** — aggregated numeric measurements over time. "Requests per second", "p99 latency", "CPU utilization". Cheap to store, fast to query, perfect for alerting and dashboards. Bad at answering "*why*" — they tell you *something* is wrong, not what.
- **Logs** — discrete events emitted by the application. "User 42 attempted to log in and got error E_INVALID_TOKEN at 14:32:01.245." Rich detail per event, expensive to store at high volume, perfect for after-the-fact investigation. Bad at showing aggregate trends without further processing.
- **Traces** — the path a single request takes through the system. "This user's request hit the gateway, called user-service (3ms), then order-service (45ms, of which 40ms was a database call), then returned." Perfect for understanding distributed behavior. Sampled in production because full tracing has too much overhead.

Each pillar answers different questions. **Metrics tell you something's wrong.** **Logs tell you what specifically happened.** **Traces tell you where the time went.** Modern systems use all three; they're complementary, not alternatives.

A useful framing: **monitoring** is "predicting what could go wrong and alerting on it" (you decide in advance what to watch). **Observability** is "being able to investigate what *did* go wrong" without pre-planning. Metrics + alerts is monitoring. The full pillar stack is observability — you can ask new questions about past behavior after the fact.


## Metrics — RED, USE, and what to measure

Two well-known method names for picking which metrics matter:

### The RED method (for request-driven services)

For every service, track:

- **R**ate — requests per second.
- **E**rror rate — errors per second (or as a fraction of rate).
- **D**uration — request latency, reported as percentiles (p50, p99).

These three answer "is the service healthy from the caller's perspective?" Almost all alerting on application services comes from RED metrics.

### The USE method (for resources)

For every resource (CPU, memory, disk, network, thread pool, connection pool), track:

- **U**tilization — % of time the resource was busy.
- **S**aturation — how much queueing is happening (length of the work queue, run queue length).
- **E**rrors — error count for the resource (failed disk reads, dropped packets).

These three answer "is the underlying infrastructure healthy?" CPU at 90% is utilization. The run queue averaging 12 threads waiting is saturation. The two together tell a more complete story than either alone — a service can have low CPU and high saturation (waiting on a slow downstream) or high CPU and low saturation (genuinely working).

### What to alert on

The biggest mistake in metrics is **alerting on causes, not symptoms.** "CPU is at 95%" might be fine — it's getting work done. **"User-facing error rate is at 5%"** is always bad. Alert on user-impacting symptoms; investigate causes after the alert fires.

The current best practice is to define **Service Level Objectives** (SLO — covered next) and alert on the **rate of error-budget consumption**, not on raw thresholds. An alert that fires when you're burning your monthly error budget twice as fast as planned is signal. An alert that fires when CPU briefly spikes is noise.


## Logs and traces — structure matters

Two patterns are essential for modern observability:

### Structured logging

Logs should be **structured** (JSON, key-value, protobuf) — not free-text. A grep over a million lines of "User logged in" is much worse than a query against a million events with `event=login user_id=42 method=oauth`. Structured logs feed dashboards, alerts, and ad-hoc queries.

Every log line should carry:

- A **correlation ID** (also called request ID, trace ID) propagated through every service for one request. This is what lets you reconstruct what a single request did across many services.
- The **service name**, **environment** (prod/staging), **deploy version**.
- The **user** and any other context that helps with investigation. (Mind PII rules — log user IDs, not emails.)

```
   {"ts": "2025-06-08T14:32:01.245Z", "level": "WARN",
    "service": "order-svc", "env": "prod", "version": "v2.14.3",
    "trace_id": "abc123", "user_id": 42,
    "event": "payment_retry", "attempt": 2, "downstream": "stripe"}
```

### Distributed tracing

A trace is a tree of **spans**, each representing a unit of work (an HTTP call, a database query, a function). Each span records start time, duration, parent span, plus arbitrary attributes. Together they tell you where a request spent its time.

The protocol that won is **OpenTelemetry**. SDKs in every language emit spans; collectors aggregate; backends (Jaeger, Tempo, AWS X-Ray, Datadog APM) store and visualize. Tracing is the only practical way to debug latency or correctness in microservices — when a request crosses ten services, no single service's logs tell you the whole story.

**Sampling matters.** Full tracing for every request is too expensive — both in network overhead and in storage. Most production systems sample 1-10% of traffic, with rules to always trace errors and slow requests.


## SLIs, SLOs, and error budgets

The Google SRE book formalized a small vocabulary that has become standard for measuring reliability:

- **SLI** — Service Level Indicator. A measured property of the service. "Fraction of requests served in under 200ms." "Fraction of requests returning 200." Specific, quantitative.
- **SLO** — Service Level Objective. The target for an SLI. "99% of requests served in under 200ms over a rolling 30 days." "99.9% of requests successful over a rolling 30 days."
- **SLA** — Service Level Agreement. The contractual version of an SLO with penalties attached. Usually weaker than the SLO ("we promise customers 99.9%, our internal SLO is 99.95% to leave headroom").
- **Error budget** — the inverse of the SLO. A 99.9% SLO permits 0.1% errors. Over 30 days, that's 43 minutes of allowed downtime — your error budget.

The error budget is the unifying idea. **You spent your error budget when something goes wrong** — an outage, a buggy deploy, a slow downstream. As long as the budget is intact, you can ship aggressively; if you're burning it fast, you slow down deploys and prioritize reliability work.

**The cultural value** is settling the eternal tension between "ship faster" (product) and "be more reliable" (operations). With an SLO and a budget, the conversation has data. "We're at 0.07% errors this month, well under budget — we can ship this feature." Or "We've burned 80% of our budget this week — no new features until we fix the cause."

**Picking SLO targets.** The honest answer is **the smallest target you can defend to your stakeholders**, not "as many nines as possible." Every additional nine costs roughly 10x more. Five nines (99.999%, 5 minutes/year) is achievable but enormously expensive. Most products are well-served by three or four nines for user-facing services, with critical paths at four. Internal services can often run at two or three nines without anyone noticing.


## Deployment strategies — shipping without downtime

The naïve deploy — stop the service, swap binaries, start it again — was acceptable in the era when a service ran on one machine. At scale and with HA expectations, every modern system uses one of these strategies.

### Rolling deployment

Update servers a few at a time. Take 10% of the fleet offline, deploy new version, return to service, wait for them to be healthy, then take the next 10%. By the end, the whole fleet is on the new version.

- **Strengths:** simple, used by every orchestrator (Kubernetes, ECS, Nomad) by default, no extra infrastructure.
- **Weaknesses:** old and new versions are running concurrently — must be backward/forward compatible. Rollback is slow (another rolling deploy). Bad deploys spread to the whole fleet before being detected.

### Blue-green deployment

Run two complete copies of the fleet: blue (current) and green (new). The load balancer points all traffic at blue. Deploy the new version to green, verify it works, then atomically swap the load balancer to point at green. Blue becomes the "rollback environment."

- **Strengths:** instant rollback (swap LB back). Easy to verify the new version before any traffic hits it.
- **Weaknesses:** double the infrastructure (briefly). Two database schemas must coexist during cutover (or you accept brief downtime for migrations).

### Canary deployment

Deploy the new version to a small slice of traffic — 1%, then 5%, then 25%, then 100%. Watch the SLIs at each stage. If error rate or latency degrades on the canary, automatically roll back without affecting the rest of the fleet.

- **Strengths:** new code is exposed to real production traffic in small doses; problems are caught with minimal blast radius. The current best-practice for high-stakes services.
- **Weaknesses:** requires sophisticated traffic routing and automated rollback. Long deploys (canary at 1% for an hour, 5% for an hour, etc.).

### Feature flags — orthogonal to all of the above

A **feature flag** is a runtime switch in the application code that turns a new code path on or off. Deploy disabled, enable for 1% of users, expand to 100%, remove the flag.

Feature flags **decouple deploy from release**. You can ship code to production behind a flag, leave it dormant for a week, then enable it without redeploying. Rollback is flipping the flag off — much faster than redeploying. Combine flags with canary deploys: deploy at low risk (the flag is off for almost everyone), then turn the flag on at canary percentages.

The cost: every flag adds branching to the code. Long-lived flags become technical debt. Treat flags as expiring — every flag should have an owner, an expected lifetime, and a removal plan. LaunchDarkly, Statsig, and homegrown systems all manage this.

### Picking a deploy strategy

**Rolling** is the default for stateless services where backward compatibility is easy. **Canary** is the upgrade when correctness is critical and you have the tooling to support it. **Blue-green** is right when you need instant rollback above all else, or for stateful services with infrastructure-level changes (DB schema, network config). **Feature flags** are orthogonal to all of them; almost every team should use flags.


## Failure modes and graceful degradation

A system has two kinds of failure: **hard failure** (it returns an error or doesn't respond) and **degraded mode** (it returns a partial or worse-than-usual response). The second is almost always better than the first.

**Graceful degradation** means designing each capability so that, when its dependencies fail, the service returns a *worse but still useful* response instead of a hard error.

Examples:

- **Recommendation engine is down on the product page.** Return the product without recommendations (show a generic "you might also like" instead). The page still loads.
- **Personalization service is slow.** Render the page with a generic homepage instead of personalized; come back to personalized when the service recovers.
- **One of three replicas of a database is unreachable.** Read from one of the remaining two; reads still succeed.
- **Cache is down.** Fall back to the database directly; slower, but correct.

**The pattern.** For every dependency call, ask "what do I return if this fails?" The answer should rarely be "an error to the user." Sometimes that's all that's possible (the user's auth check legitimately can't proceed without auth), but most calls have a sensible fallback.

**The mental discipline.** "What happens when X fails?" should be answerable for every X in the system. The answer becomes part of the design, not an afterthought during the incident. Chaos engineering (deliberately failing things in production to verify the graceful-degradation paths work) is the operationalization of this.

**Hot tip.** Graceful degradation has to be **tested**. A fallback path that nobody has exercised since the day it was written will almost certainly not work when needed. Run it deliberately — in tests, in chaos experiments, in periodic "failure drills" — to confirm it works and to keep it warm in the team's collective memory.


## Load shedding and backpressure

When demand exceeds capacity, the system has two options:

- **Try to serve every request.** Queues grow. Latency increases. Eventually requests time out or memory fills up. The system crashes or degrades for everyone.
- **Reject some requests immediately.** The accepted requests are served at normal latency. The rejected ones fail fast — caller can retry, fall back, or show an error.

The second option is **load shedding**. It's counterintuitive — actively rejecting work — but it's the only stable behavior under overload. A system that tries to serve everyone collapses; a system that sheds load stays up for those it serves.

### Two flavours

**Random shedding.** Reject a percentage of incoming requests. Simple, fair, but ignores priority — a paying customer's request and an internal health check have the same chance of being dropped.

**Priority shedding.** Categorize requests by priority (paying customer > free user > internal report > debug request). Drop low-priority first. Requires the application to expose request priority, which most don't by default.

### Where to shed

**At the edge** is the cheapest. The API gateway or load balancer rejects with a 429 (Too Many Requests) or 503 (Service Unavailable). The backend never sees the rejected request, so cost is minimal.

**At the application** is more precise but more expensive — the request has already traveled to your service before you reject it. Useful when only the application knows the request's priority.

**At the database / queue** is critical for downstream protection. A queue with a bounded depth automatically rejects when full, which becomes a backpressure signal to upstreams.

### Backpressure — pushing the signal up the chain

When a downstream is full, the upstream needs to know. **Backpressure** is the explicit feedback signal: "I can't take more right now." Mechanisms:

- **HTTP 429** with a `Retry-After` header.
- **gRPC RESOURCE_EXHAUSTED** status with backoff hints.
- **Queue depth** monitored by producers — if the queue is filling, slow down or block.
- **Reactive Streams** semantics in async libraries (Project Reactor, RxJava) — consumers request what they can handle.

Without backpressure, a fast producer plus a slow consumer is a guaranteed memory leak in the queue between them. Backpressure converts the failure mode from "OOM" to "explicit slowdown" — much easier to diagnose and recover from.

### The hard tradeoff

Load shedding inherently means *some users see errors*. The skill is choosing **which users**. Random shedding affects everyone uniformly; priority shedding lets you keep paying customers happy while degrading free-tier traffic. Match the strategy to the business model.


## Putting it together — the operations recipe

A complete operations posture for a modern microservices system, in one paragraph:

> Every service emits **structured logs with correlation IDs**, **metrics for RED + USE**, and **traces** sampled at 1-10% (always for errors). **SLOs** are defined per user-facing endpoint, alerts fire on **error-budget burn rate**, not raw thresholds. Deploys use **canary** with automated rollback on SLI degradation; risky changes hide behind **feature flags**. Every dependency call has a **timeout, retry with backoff+jitter, circuit breaker, bulkhead, and fallback**. The fallback paths are exercised by **chaos experiments** quarterly. When capacity is exceeded, the system **sheds load** by priority rather than collapsing. **Post-incident reviews** are blameless; the output is concrete action items with owners and dates, tracked to completion.

That's the recipe. Each piece is straightforward. The skill is operating all of it consistently and updating it as the system evolves.


Notebook eight closes the series with **Case Studies** — five canonical system design problems worked end-to-end: URL shortener, rate limiter, Twitter-style feed, chat system, and distributed cache. Each one starts from requirements and capacity estimates, sketches the data model, walks the request path, and calls out the trade-offs we made along the way. The vocabulary from notebooks one through seven becomes the design language for all five.
